# VietDub experimental Adam TTS 50h — GPU vocoder reconstruction

**Optional source:** [Phitran21 Vietnamese-Adam-TTS-50h](https://huggingface.co/datasets/Phitran21/Vietnamese-Adam-TTS-50h) is **synthetic**, one Adam male voice preset, not recordings of a consenting human voice. CC BY 4.0 on the dataset card does **not** settle upstream generation model and voice-asset rights. This notebook does not use VieNeu or any pretrained TTS model.

**Preconditions:** After reviewing both sets of voice rights, choose `DATA_INPUT_MODE=remote` to stream up to 500 selected clips over bounded HF HTTP Range without downloading the ~17.5 GB ZIP, or `local_zip` with a privately mounted full archive for >500 clips. Neither mode bypasses source-rights review. This notebook **has not been executed on Kaggle or validated for real-audio quality**. Training a vocoder is **not** text-to-speech.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO = Path('/kaggle/working/vietdub-tts')
DATA_INPUT_MODE = 'remote'  # 'remote' <=500 clips via HTTP Range; 'local_zip' for 50h
ARCHIVE = Path('/kaggle/input/vietnamese-adam-tts-50h/finnal_tts.zip')  # Optional local ZIP
PREPARED = Path('/kaggle/working/vietdub-adam-prepared')
TRAIN_OUTPUT = Path('/kaggle/working/vietdub-adam-vocoder-run')

DATASET_RIGHTS_REVIEWED = False        # Human-reviewed dataset terms
UPSTREAM_VOICE_RIGHTS_REVIEWED = False # Independently reviewed Adam voice/generation terms
APPROVE_VOCODER_TRAINING = False       # Must be an affirmative operator decision
MAX_CLIPS = 250 if DATA_INPUT_MODE == 'remote' else 2000
TOTAL_TRAIN_STEPS = 50000
CHECKPOINT_EVERY = 250

assert DATA_INPUT_MODE in ('remote', 'local_zip')
if DATA_INPUT_MODE == 'local_zip':
    assert ARCHIVE.is_file(), f'Attach finnal_tts.zip as a Kaggle input: {ARCHIVE}'
    print('Source size (GB):', round(ARCHIVE.stat().st_size / 1e9, 2))
else:
    print('Remote bounded HTTP Range mode; no 17.5GB source download')
assert DATASET_RIGHTS_REVIEWED and UPSTREAM_VOICE_RIGHTS_REVIEWED and APPROVE_VOCODER_TRAINING, (
    'Dataset CC BY label is not sufficient. Confirm the separate upstream synthetic voice rights.')


In [ ]:
import torch
assert torch.cuda.is_available(), 'Enable GPU accelerator in Kaggle settings'
if not REPO.is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'feat/native-hifigan-vocoder-superpowers',
                    'https://github.com/hungtvb/vietdub-tts.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
from vietdub_native.adam_tts_vocoder import prepare_adam_tts
from vietdub_native.adam_remote_probe import prepare_remote_adam_tts
if not (PREPARED / 'vocoder.jsonl').is_file():
    if DATA_INPUT_MODE == 'remote':
        result = prepare_remote_adam_tts(PREPARED,
            max_clips=MAX_CLIPS, max_transfer_bytes=2 * 1024**3,
            rights_reviewed=DATASET_RIGHTS_REVIEWED,
            upstream_voice_rights_reviewed=UPSTREAM_VOICE_RIGHTS_REVIEWED,
            training_approved=APPROVE_VOCODER_TRAINING)
    else:
        result = prepare_adam_tts(ARCHIVE, PREPARED,
            max_clips=MAX_CLIPS,
            rights_reviewed=DATASET_RIGHTS_REVIEWED,
            upstream_voice_rights_reviewed=UPSTREAM_VOICE_RIGHTS_REVIEWED,
            training_approved=APPROVE_VOCODER_TRAINING)
else:
    result = json.loads((PREPARED / 'import_report.json').read_text(encoding='utf-8'))
assert result['train_dev_policy'] == 'text_group_disjoint_single_synthetic_voice'
assert result['training_approved_by_operator'] is True
print(json.dumps(result, indent=2, ensure_ascii=False))


In [ ]:
from vietdub_native.hifigan_train import train_hifigan
resume = (TRAIN_OUTPUT / 'private_checkpoints/training_state.pt').is_file()
report = train_hifigan(PREPARED / 'vocoder.jsonl', TRAIN_OUTPUT,
                      steps=TOTAL_TRAIN_STEPS, device='cuda', crop_frames=48,
                      checkpoint_every=CHECKPOINT_EVERY, resume=resume, eval_seconds=5)
print(json.dumps(report, indent=2, ensure_ascii=False))
assert report['intelligibility_verified'] is False


In [ ]:
from IPython.display import Audio, display
# A generated waveform reconstructed from held-out synthetic Adam text/audio.
# No original voice recording or synthetic dataset audio is published.
reconstructed = TRAIN_OUTPUT / 'dev_reconstructed.wav'
assert reconstructed.is_file()
display(Audio(filename=str(reconstructed)))
print('Listen privately against the matched original, then report quality honestly.')
print('Checkpoint contains rights-dependent weights: store privately, do not attach publicly.')
